In [3]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')

True

In [4]:
import pyarrow as pa
import pandas
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.exceptions import NamespaceAlreadyExistsError
from pyiceberg.transforms import IdentityTransform, DayTransform

/Users/jhonsfran/repos/unprice/internal/lakehouse/lakenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
# 1. Connect to R2 Data Catalog

# Define catalog connection details
ENV = "dev"
CATALOG = "unprice-lakehouse-" + ENV
ACCOUNT_ID  = os.environ['ACCOUNT_ID']
CATALOG_URI = "https://catalog.cloudflarestorage.com/" + ACCOUNT_ID + "/" + CATALOG
TOKEN       = os.environ['TOKEN']
WAREHOUSE   = ACCOUNT_ID + "_" + CATALOG

# Connect to R2 Data Catalog
catalog = RestCatalog(
    name="unprice",
    warehouse=WAREHOUSE,
    uri=CATALOG_URI,
    token=TOKEN,
)

# list namespaces
print(catalog.list_namespaces())

[('lakehouse',)]


In [11]:
# drop tables
# catalog.drop_table('lakehouse.metadata')
# catalog.drop_table('lakehouse.entitlement_snapshot')
# catalog.drop_table('lakehouse.usage')
# catalog.drop_table('lakehouse.verification')

In [12]:
# List tables in the "default" namespace
tables = catalog.list_tables(namespace='lakehouse')
print(tables)

[('lakehouse', 'metadata'), ('lakehouse', 'usage'), ('lakehouse', 'entitlement_snapshot'), ('lakehouse', 'verification')]


In [13]:
# Load and print schema for every table in the lakehouse namespace
tables = catalog.list_tables(namespace='lakehouse')
for table_id in tables:
    table = catalog.load_table(table_id)
    print(f"--- {table_id[0]}.{table_id[1]} ---")
    print(table.schema())
    print()

--- lakehouse.metadata ---
table {
  1: __ingest_ts: required timestamp
  2: id: required string
  3: event_date: required string
  4: project_id: required string
  5: customer_id: required string
  6: schema_version: required long
  7: timestamp: required long
  8: payload: optional string
}

--- lakehouse.usage ---
table {
  1: __ingest_ts: required timestamp
  2: id: required string
  3: event_date: required string
  4: request_id: required string
  5: project_id: required string
  6: customer_id: required string
  7: timestamp: required long
  8: idempotence_key: required string
  9: entitlement_id: required string
  10: feature_slug: required string
  11: schema_version: required long
  12: usage: optional double
  13: allowed: optional boolean
  14: deleted: optional long
  15: meta_id: optional string
  16: country: optional string
  17: region: optional string
  18: action: optional string
  19: key_id: optional string
  20: unit_of_measure: optional string
  21: cost: optional

In [22]:
# List and peek contents of every table in the lakehouse namespace
import pandas as pd

tables = catalog.list_tables(namespace='lakehouse')
MAX_ROWS = 10  # rows to show per table

for table_id in tables:
    name = f"{table_id[0]}.{table_id[1]}"
    table = catalog.load_table(table_id)
    scan = table.scan(limit=MAX_ROWS)
    df = scan.to_pandas()
    print(f"\n=== {name} (showing up to {MAX_ROWS} rows) ===")
    if df.empty:
        print("(no rows)")
    else:
        display(df)


=== lakehouse.metadata (showing up to 10 rows) ===


,__ingest_ts,id,event_date,project_id,customer_id,schema_version,timestamp,payload
0,2026-02-16 21:14:15,10071221946347133000,2026-02-16,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEf2K2jxSUmeYyA1CX,1,1771276448968,"{""source"":""asdasdasd""}"
1,2026-02-16 21:12:15,10071221946347133000,2026-02-16,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1,1771276329630,"{""source"":""asdasdasd""}"



=== lakehouse.usage (showing up to 10 rows) ===


,__ingest_ts,id,event_date,request_id,project_id,customer_id,timestamp,idempotence_key,entitlement_id,feature_slug,...,deleted,meta_id,country,region,action,key_id,unit_of_measure,cost,rate_amount,rate_currency
0,2026-02-16 21:08:02,01KHM4AEZHNEZ51KKP6HWJZQDT,2026-02-16,req_11Tvr7j4Ed41F5AQRV6wZc,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771275893553,123e4567-e89b-12d3-a456-426614174000:177127589...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
1,2026-02-16 21:08:02,01KHM4AEZFMCQ78A17DGMR87G9,2026-02-16,req_11Tvr7iyXMVLStXzUyYxs5,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771275893533,123e4567-e89b-12d3-a456-426614174000:177127589...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
2,2026-02-16 21:08:09,01KHM4G8J7KKJY2737P3BJK1DS,2026-02-16,req_11TvrMkJY1NrXxURXCQQCv,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276083766,123e4567-e89b-12d3-a456-426614174000:177127608...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
3,2026-02-16 21:08:25,01KHM4GT26NSH1PSM4VC714Z82,2026-02-16,req_11TvrP4xyRFSPQrDjDMEKP,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276101697,123e4567-e89b-12d3-a456-426614174000:177127610...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
4,2026-02-16 21:08:25,01KHM4GSXTX05VN6E6CY14HKH6,2026-02-16,req_11TvrP4NWFB6Meic3gTZtc,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276101558,123e4567-e89b-12d3-a456-426614174000:177127610...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
5,2026-02-16 21:08:25,01KHM4GSRVZFG95RXSZGZYBS0F,2026-02-16,req_11TvrP3gq3AAkDeZ8dho23,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276101398,123e4567-e89b-12d3-a456-426614174000:177127610...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
6,2026-02-16 21:08:25,01KHM4GSKXRF30WDD3B79VN0G0,2026-02-16,req_11TvrP31QDKzq9FCcn4V7E,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276101239,123e4567-e89b-12d3-a456-426614174000:177127610...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
7,2026-02-16 21:08:25,01KHM4GSF505Z6WTKVJCQ50PC1,2026-02-16,req_11TvrP2NCqBvH81c47r4Qp,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276101089,123e4567-e89b-12d3-a456-426614174000:177127610...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
8,2026-02-16 21:08:25,01KHM4GSAT3QJ00SDGYHRRTMQA,2026-02-16,req_11TvrP1mEtjeMhYV4qPLQh,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276100948,123e4567-e89b-12d3-a456-426614174000:177127610...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
9,2026-02-16 21:08:25,01KHM4GS6FKGBX7MB124Y44GXQ,2026-02-16,req_11TvrP1BWsDyCt9LyCKdV4,proj_11STWG6AokEni2F3eQugHb,cus_11TqNFStayKrDBz5JFCzAC,1771276100812,123e4567-e89b-12d3-a456-426614174000:177127610...,ent_11TrbKSNxVS8eVV2VKKSrE,events,...,0,8147303860956607000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR



=== lakehouse.entitlement_snapshot (showing up to 10 rows) ===


,__ingest_ts,id,event_date,project_id,customer_id,schema_version,timestamp,feature_slug,feature_type,unit_of_measure,reset_config,aggregation_method,merging_policy,limit,effective_at,expires_at,version,grants,metadata
0,2026-02-16 21:14:15,ent_11TqNEiMEehJjUMoxWAq7g,2026-02-13,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEf2K2jxSUmeYyA1CX,1,1771025779100,events,usage,units,"{""name"":""monthly"",""resetInterval"":""month"",""res...",sum,sum,1000,1771025778327,NaN,mvee8BMh4o7xstchtqYBHaUqH+D8zm+wW4XTDd0sLNs=,"[{""id"":""grnt_11TqNEfJgH8ASmDZUrsgkW"",""type"":""s...","{""realtime"":false,""notifyUsageThreshold"":95,""o..."



=== lakehouse.verification (showing up to 10 rows) ===


,__ingest_ts,id,event_date,project_id,entitlement_id,feature_slug,customer_id,request_id,timestamp,schema_version,...,region,meta_id,action,key_id,usage,remaining,unit_of_measure,cost,rate_amount,rate_currency
0,2026-02-16 21:14:15,1,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrpfhU9ceiS2uHneTcp,1771276448968,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
1,2026-02-16 21:14:15,2,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrpgbYV1gYBejTQ2G7e,1771276449178,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
2,2026-02-16 21:14:15,3,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrphGEAH2EUJ1PNt8QE,1771276449334,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
3,2026-02-16 21:14:15,4,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tvrpi75XEfWrMgXrPsz6,1771276449531,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
4,2026-02-16 21:14:15,5,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrpivSb4wLPLfuyTyZ2,1771276449722,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
5,2026-02-16 21:14:15,6,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrpjjJtXKEyaeaVQBhH,1771276449911,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
6,2026-02-16 21:14:15,7,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrpkVhN738CkRwTbmWq,1771276450090,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
7,2026-02-16 21:14:15,8,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrpmEMANc6V5aQpsdsS,1771276450262,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
8,2026-02-16 21:14:15,9,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tvrpn3DTq32AhJNAyPy4,1771276450451,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None
9,2026-02-16 21:14:15,10,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvrpngQqy9jm6FWQxuH1,1771276450601,1,...,TXL,10071221946347133000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,1000,None,NaN,NaN,None


In [63]:
# 1. Create namespace
# catalog.create_namespace("default")

In [23]:
# 2. Load the pipeline-created table
verifications = catalog.load_table("lakehouse.verification")
print("Current Spec:", verifications.spec())
metadata = catalog.load_table("lakehouse.metadata")
print("Current Spec:", verifications.spec())
entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
print("Current Spec:", verifications.spec())
usage = catalog.load_table("lakehouse.usage")
print("Current Spec:", verifications.spec())

Current Spec: [
  1000: __ingest_ts_day: day(1)
]
Current Spec: [
  1000: __ingest_ts_day: day(1)
]
Current Spec: [
  1000: __ingest_ts_day: day(1)
]
Current Spec: [
  1000: __ingest_ts_day: day(1)
]


In [24]:
# verifications
# Make sure to use the namespace defined in the pipeline (often 'default')
verifications = catalog.load_table("lakehouse.verification")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)
        
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")
    update.add_field("event_date", DayTransform(), "date_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

ValueError: day cannot transform string values from event_date

In [21]:
# List tables in the "default" namespace
new_tables = catalog.list_tables(namespace='default')
print("Tables in 'default':", new_tables)

Tables in 'default': [('default', 'prueba')]


In [22]:
# Load a specific table
table = catalog.load_table(('default', 'prueba'))

# Print the schema
print(table.schema())

table {
  1: event_time: required timestamp
  2: user_id: required string
  3: event_data: optional string
}
